# Logging


*Setting up logging can be hard, but in this notebook we will go through different tools and libraries that can help you set up logging in your application.*

### What is logging?

- Logging is a means of tracking "events" when your application runs. 

- An *event* can be anything of interest that happens during the execution of your program like occurance of an error, a simple infomatic message like your program started or you API call was successful etc.

  - Events are logged with a descriptive message which optionally can have associated application data.

  - Events also have an importance which you as the developer ascribes it; the importance can also be called the **level or severity**.


### When to use logging?

| Task you want to perform                                                                 | The best tool for the task                                                                                         |
|------------------------------------------------------------------------------------------|--------------------------------------------------------------------------------------------------------------------|
| Display console output for ordinary usage of a command line script or program            | `print()`                                                                                                          |
| Report events that occur during normal operation of a program (e.g. for status monitoring or fault investigation) | A logger’s `info()` (or `debug()` method for very detailed output for diagnostic purposes)                          |
| Issue a warning regarding a particular runtime event                                     | `warnings.warn()` in library code if the issue is avoidable and the client application should be modified to eliminate the warning<br>A logger’s `warning()` method, if there is nothing the client application can do about the situation, but the event should still be noted |
| Report an error regarding a particular runtime event                                     | Raise an exception                                                                                                 |
| Report suppression of an error without raising an exception (e.g. error handler in a long-running server process) | A logger’s `error()`, `exception()`, or `critical()` method as appropriate for the specific error and application domain |


### Logging Levels

The logger methods are named after the *level or severity* of the events they are used to track. The standard levels (in increasing order of severity) are: `DEBUG`, `INFO`, `WARNING`, `ERROR`, and `CRITICAL`. 

You can read more about them [*here*](https://docs.python.org/3/library/logging.html#logging-levels).


<p align="right"><i>Source: <a href="https://docs.python.org/3/howto/logging.html">Python Logging Docs</a></i></p>

In [45]:
# Install the logging libraries

%pip install -q loguru aws-lambda-powertools mangum nest_asyncio

Note: you may need to restart the kernel to use updated packages.


# TODO

- [x] How to toggle the verbosity of traceback in loguru?
- [x] Ideally show the both ways

- [ ] Log Perf. Analysis: Time analysis

## Python's `logging` module

In [1]:
# Setup logging in the simplest way possible

import os
import sys
import logging


# get log level from environment variable
LOG_LEVEL = os.environ.get("LOG_LEVEL", "INFO").upper()

# configure logger object with the desired log level and format
logging.basicConfig(
    format="{asctime} | {levelname} | {name}:{lineno}:{funcName} | {message}",
    style="{",  # uses {} as placeholders
    level=LOG_LEVEL,
    stream=sys.stdout,  # where to write the log messages, in this case stdout or console
)

# create logger object with the name of the current module/file to start logging
logger = logging.getLogger(__name__)

# logger = logging.getLogger("my_logger_instance")  # you can also use a custom name

# log some messages
logger.debug("This is a debug message")  # this will not be printed because the log level is set to INFO
logger.info("This is an info message")
logger.warning("This is a warning message")
logger.error("This is an error message")
logger.critical("This is a critical message")
# logger.exception("This is an exception message")

2024-08-13 14:04:01,858 | INFO | __main__:26:<module> | This is an info message
2024-08-13 14:04:01,859 | WARNING | __main__:27:<module> | This is a warning message
2024-08-13 14:04:01,860 | ERROR | __main__:28:<module> | This is an error message
2024-08-13 14:04:01,861 | CRITICAL | __main__:29:<module> | This is a critical message


The logging library takes a modular approach and offers several categories of components: *loggers, handlers, filters, and formatters*.

- [***Loggers***](https://docs.python.org/3/howto/logging.html#loggers) expose the interface that application code directly uses.
- [***Handlers***](https://docs.python.org/3/howto/logging.html#handlers) send the log records (created by loggers) to the appropriate destination.
- [***Filters***](https://docs.python.org/3/library/logging.html#filter-objects) provide a finer grained facility for determining which log records to output.
- [***Formatters***](https://docs.python.org/3/library/logging.html#logrecord-attributes) specify the layout of log records in the final output.


Log event information is passed between *loggers, handlers, filters and formatters* in a LogRecord instance.Logging is performed by calling methods on instances of the Logger class. You can read more on it [*here*](https://docs.python.org/3/howto/logging.html#advanced-logging-tutorial) and [*here*](https://docs.python.org/3/library/logging.html).



<p align="right"><i>Source: <a href="https://docs.python.org/3/howto/logging.html#advanced-logging-tutorial">Advanced Python Logging Docs</a></i></p>

In [1]:
# Advanced logging setup with multiple handlers and a formatter
# Ref: https://docs.python.org/3/howto/logging-cookbook.html#logging-cookbook

import logging
import sys


# create a console handler and set its log level
ch = logging.StreamHandler(stream=sys.stderr)
ch.setLevel(logging.DEBUG)  # log all messages

# create another handler with a different log level that logs to a file
fh = logging.FileHandler("demo.log", mode="w", encoding="utf-8")
fh.setLevel(logging.WARNING)  # only log messages with WARNING level or higher

# create a formatter
formatter = logging.Formatter("{asctime} | {levelname} | {name}:{lineno}:{funcName} | {message}", style="{")
ch.setFormatter(formatter)  # add the formatter to the console handler
fh.setFormatter(formatter)  # add the formatter to the file handler


# create the logger
logger = logging.getLogger(__name__)

# add the handlers to the logger object
logger.addHandler(ch)
logger.addHandler(fh)

# log some messages
logger.debug("This is a debug message")
logger.info("This is an info message")
logger.warning("This is a warning message")
logger.error("This is an error message")
logger.critical("This is a critical message")


try:
    1 / 0
except Exception as e:
    logger.exception(
        "Exception details without traceback: %s", e, exc_info=False
    )  # this will log the exception message
    logger.exception(
        "Exception details with traceback: %s", e, exc_info=True
    )  # this will log the exception message and traceback

2024-08-13 15:11:22,528 | WARNING | __main__:32:<module> | This is a warning message
2024-08-13 15:11:22,529 | ERROR | __main__:33:<module> | This is an error message
2024-08-13 15:11:22,531 | CRITICAL | __main__:34:<module> | This is a critical message
2024-08-13 15:11:22,532 | ERROR | __main__:40:<module> | Exception details without traceback: division by zero
2024-08-13 15:11:22,533 | ERROR | __main__:41:<module> | Exception details with traceback: division by zero
Traceback (most recent call last):
  File "/tmp/ipykernel_121362/3703661617.py", line 38, in <module>
    1 / 0
    ~~^~~
ZeroDivisionError: division by zero


## [`loguru`](https://loguru.readthedocs.io/en/stable/overview.html)

>The main concept of Loguru is that there is one and only one logger. No Handler, no Formatter, no Filter: one function to rule them all.

In [2]:
from loguru import logger

# log some messages
logger.debug("This is a debug message")
logger.info("This is an info message")
logger.warning("This is a warning message")
logger.error("This is an error message")
logger.critical("This is a critical message")
# logger.exception("This is an exception message")

2024-08-13 14:35:17.036 | DEBUG    | __main__:<module>:4 - This is a debug message
2024-08-13 14:35:17.037 | INFO     | __main__:<module>:5 - This is an info message
2024-08-13 14:35:17.038 | WARNING  | __main__:<module>:6 - This is a warning message
2024-08-13 14:35:17.038 | ERROR    | __main__:<module>:7 - This is an error message
2024-08-13 14:35:17.039 | CRITICAL | __main__:<module>:8 - This is a critical message


In [1]:
import os
import sys
import json
import loguru
from loguru import logger

os.environ["LOGURU_LEVEL"] = "INFO"  # set the log level to INFO


def serialize_extra_keys(record):
    extra = record["extra"]
    if extra:
        record["extra"] = json.dumps(extra)

    return record


logging_config = {
    "handlers": [
        {
            "sink": sys.stdout,
            "format": "<green>{time:YYYY-MM-DD HH:mm:ss}</green> | <level>{level: <8}</level> | <cyan>{name}:{function}:{line}</cyan> | <level>{message}</level> | <level>{extra}</level>",
            "level": os.environ["LOGURU_LEVEL"],
            "filter": serialize_extra_keys,
            "colorize": True,
        },
    ],
}

# Remove the default logger config and add custom configurations
logger.remove()
logger.configure(**logging_config)


# log some messages
logger.debug("This is a debug message")
logger.success("This is a success message")
logger.info("This is an info message")
logger.warning("This is a warning message")
logger.error("This is an error message")
logger.critical("This is a critical message")
logger.info(
    "Adding extra data to log messages",
    extra={"extra_key": "extra_value", "another_key": "another_value", "user_id": 12345},
)


# https://loguru.readthedocs.io/en/stable/overview.html#lazy-evaluation-of-expensive-functions
# https://github.com/Delgan/loguru/issues/30

try:
    1 / 0
except Exception as e:
    # logger.exception("An exception occurred: {}", e)    # log the exception details & traceback
    logger.opt(exception=False).error("Exception details without traceback: {}", e)
    
    # UNCOMMENT THE LINE BELOW TO SEE THE FULL TRACEBACK: loguru gives a ver long traceback
    # logger.opt(exception=True).error("Exception details with traceback: {}", e)

2024-08-13 20:14:34 | SUCCESS  | __main__:<module>:37 | This is a success message | {}
2024-08-13 20:14:34 | INFO     | __main__:<module>:38 | This is an info message | {}
2024-08-13 20:14:34 | WARNING  | __main__:<module>:39 | This is a warning message | {}
2024-08-13 20:14:34 | ERROR    | __main__:<module>:40 | This is an error message | {}
2024-08-13 20:14:34 | CRITICAL | __main__:<module>:41 | This is a critical message | {}
2024-08-13 20:14:34 | INFO     | __main__:<module>:42 | Adding extra data to log messages | {"extra": {"extra_key": "extra_value", "another_key": "another_value", "user_id": 12345}}
2024-08-13 20:14:34 | ERROR    | __main__:<module>:55 | Exception details without traceback: division by zero | {}


## [`aws-lambda-powertools`](https://docs.powertools.aws.dev/lambda/python/latest/core/logger/)

>Logging utilities for AWS Lambda functions.

- `aws_lambda_powertools` module provides `inject_lambda_context` decorator which injects AWS Lambda specific context information into the log record like `cold_start`, `function_request_id`, etc. along an option for logging incoming lambda event.

In [6]:
import os
import sys
from typing import Any
from aws_lambda_powertools import Logger
from aws_lambda_powertools.utilities.typing import LambdaContext
import os

# Powertools logger level can be configured using the environment variable POWERTOOLS_LOG_LEVEL
os.environ["POWERTOOLS_LOG_LEVEL"] = "WARNING"
os.environ["POWERTOOLS_SERVICE_NAME"] = "demo-service"

# Sets service key that will be present across all log statements
# If you have a lot of different services, being able to easily filter logs by service name will be critical when you are troubleshooting a problem in production.


logger = Logger(
    # stream=sys.stdout,
    # child=True, # If True, the logger will inherit the configuration from the parent logger into another logger instance
    # log_uncaught_exceptions=True,  # log uncaught exceptions
)


@logger.inject_lambda_context()  # clear_state=True will clear the logger state for each invocation
def handler(event: dict, context: LambdaContext) -> Any:
    try:
        logger.debug("This is a debug message")
        print()
        logger.info("This is an info message")
        print()
        logger.info("This is an info message with extra metadata", extra={"extra_key": "extra_value"})
        print()
        logger.info("This is an info message with another extra metadata", another_extra_key="another_extra_value")
        print()
        logger.warning("This is a warning message")
        print()

        # Add extra data to future log messages
        # logger.append_keys(extra_data={"user_id": 12345}); print()

        logger.error("This is an error message")
        print()
        logger.critical("This is a critical message")
        print()

        1 / 0  # raise an exception

        logger.info("Finished processing the event.")
        print()

        return {"statusCode": 200, "body": "Hello, World!"}
    except Exception as e:
        # THIS format is not supported by the logger
        # logger.exception("Exception without traceback: {}", e, exc_info=False); print()
        # logger.exception("Exception with traceback: {}", e, exc_info=True); print()

        logger.exception("Exception without traceback: %s", e, exc_info=False)
        print()
        logger.exception("Exception with traceback: %s", e, exc_info=True)
        print()
        return {"statusCode": 500, "body": "Internal Server Error"}


# Simulate the Lambda context (optional)
class Context:
    def __init__(self):
        self.function_name = "demo-function"
        self.memory_limit_in_mb = 128
        self.invoked_function_arn = "arn:aws:lambda:us-west-2:123456789012:function:test-function"
        self.aws_request_id = "fake-request-id"


if __name__ == "__main__":
    context = Context()

    # Call the handler function as Lambda would
    response = handler({}, context)
    print(response)





{"level":"WARNING","location":"handler:29","message":"This is a warning message","timestamp":"2024-08-13 20:04:33,761+0530","service":"demo-service","cold_start":false,"function_name":"demo-function","function_memory_size":128,"function_arn":"arn:aws:lambda:us-west-2:123456789012:function:test-function","function_request_id":"fake-request-id"}

{"level":"ERROR","location":"handler:34","message":"This is an error message","timestamp":"2024-08-13 20:04:33,762+0530","service":"demo-service","cold_start":false,"function_name":"demo-function","function_memory_size":128,"function_arn":"arn:aws:lambda:us-west-2:123456789012:function:test-function","function_request_id":"fake-request-id"}

{"level":"CRITICAL","location":"handler:35","message":"This is a critical message","timestamp":"2024-08-13 20:04:33,763+0530","service":"demo-service","cold_start":false,"function_name":"demo-function","function_memory_size":128,"function_arn":"arn:aws:lambda:us-west-2:123456789012:function:test-function